# RuleShift Benchmark v1 — Kaggle Benchmark

Official `kaggle_benchmarks` notebook for the RuleShift Benchmark v1 cognitive flexibility benchmark.

- **Leaderboard task:** `ruleshift_benchmark_v1_binary` — Binary (post-shift probe accuracy); evaluated over `public_leaderboard` and `private_leaderboard`
- **Supplementary:** Narrative — same leaderboard episodes, not leaderboard-scored; results included in the payload as robustness evidence only
- **Local only:** `dev` split — used for local validation and debug checks; never included in the official leaderboard evaluation
- **Metric:** Post-shift Probe Accuracy = correct probes / total probes
- **Per-episode return:** `(num_correct, 4)`

## Imports

In [ ]:
import sys
from pathlib import Path

import kaggle_benchmarks as kbench

## Setup

Resolves the repo root and adds `src/` to `sys.path`. On Kaggle, the benchmark repo is attached as an input dataset. The private evaluation dataset is a separate authorized attachment that is discovered later only when present.

In [ ]:
def _find_repo_root() -> Path:
    """Locate the repo root from the notebook's working directory or Kaggle input paths."""
    candidates: list[Path] = []
    seen: set[Path] = set()

    for origin in (Path.cwd().resolve(),):
        for candidate in (origin, *origin.parents):
            if candidate not in seen:
                seen.add(candidate)
                candidates.append(candidate)

    for search_root in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not search_root.exists():
            continue
        for manifest_path in search_root.rglob("frozen_artifacts_manifest.json"):
            candidate = manifest_path.parents[2]
            if candidate not in seen:
                seen.add(candidate)
                candidates.append(candidate)

    for candidate in candidates:
        if (candidate / "src").is_dir() and (
            candidate / "packaging" / "kaggle" / "frozen_artifacts_manifest.json"
        ).is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate repo root. Expected src/ and "
        "packaging/kaggle/frozen_artifacts_manifest.json."
    )


REPO_ROOT = _find_repo_root()

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

PRIVATE_DATASET_ROOT = None

## Load frozen splits

`dev` and `public_leaderboard` are always reconstructed from frozen seed manifests. `private_leaderboard` is discovered lazily and loaded only when the attached authorized private dataset mount is present.

In [ ]:
from core.kaggle import (
    BenchmarkRunLogger,
    BinaryResponse,
    EpisodeResultLedgerWriter,
    Label,
    NotebookStatus,
    build_run_context,
    normalize_binary_response,
    normalize_narrative_response,
    run_binary_episode,
    run_narrative_episode,
    score_episode,
    write_diagnostics_summary,
    write_run_manifest,
)
from core.splits import load_frozen_split
from core.private_split import discover_private_dataset_root, load_private_split
from core.parser import PARSER_VERSION
from core.metrics import METRIC_VERSION, compute_metrics
from core.slices import build_slice_report, compute_episode_slice_data
from tasks.ruleshift_benchmark.render import render_binary_prompt, render_narrative_prompt

_DEV_PARTITION = "dev"
_LEADERBOARD_PARTITIONS = ("public_leaderboard", "private_leaderboard")
_PUBLIC_RUNTIME_PARTITIONS = (_DEV_PARTITION, "public_leaderboard")
_PROGRESS_REFRESH_EVERY = 10
RUN_CONTEXT = build_run_context(repo_root=REPO_ROOT, llm=getattr(kbench, "llm", None))
RUN_OUTPUT_DIR = RUN_CONTEXT.output_dir
RUN_LOGGER = BenchmarkRunLogger(RUN_CONTEXT)
RUN_LOG_PATH = RUN_LOGGER.log_path
RUN_EPISODE_LEDGER = EpisodeResultLedgerWriter(RUN_CONTEXT)
RUN_EPISODE_LEDGER_PATH = RUN_EPISODE_LEDGER.path
BENCHMARK_RESULT_PATH = RUN_OUTPUT_DIR / "benchmark_result.json"
_BINARY_LOG_PHASE = "binary"
_NARRATIVE_LOG_PHASE = "narrative"


def _log_phase_start(
    phase: str,
    *,
    task_mode: str = "notebook",
    detail: str,
    total: int | None = None,
) -> None:
    RUN_LOGGER.log_phase_started(
        phase=phase,
        task_mode=task_mode,
        detail=detail,
        total=total,
    )


def _log_phase_finish(
    phase: str,
    *,
    task_mode: str = "notebook",
    detail: str,
    status: str = "completed",
    level: str = "info",
    processed: int | None = None,
    total: int | None = None,
    warnings: int = 0,
    errors: int = 0,
) -> None:
    RUN_LOGGER.log_phase_finished(
        phase=phase,
        task_mode=task_mode,
        detail=detail,
        status=status,
        level=level,
        processed=processed,
        total=total,
        warnings=warnings,
        errors=errors,
    )


def _log_exception(
    exc: BaseException,
    *,
    phase: str,
    task_mode: str,
    episode_id: str | None = None,
) -> None:
    RUN_LOGGER.log_exception(exc, phase=phase, task_mode=task_mode, episode_id=episode_id)


def _invalidate_run(*, phase: str, detail: str, reason: str) -> None:
    RUN_LOGGER.log_run_invalidated(
        phase=phase,
        detail=detail,
        reason=reason,
    )


RUN_LOGGER.log_run_started(
    output_dir=str(RUN_OUTPUT_DIR),
    repo_root=str(REPO_ROOT),
)

_bootstrap_status = NotebookStatus(refresh_every=_PROGRESS_REFRESH_EVERY)
RUN_LOGGER.log_bootstrap_started(
    detail="Locating repo root and frozen split artifacts",
    total=len(_PUBLIC_RUNTIME_PARTITIONS) + 1,
)
_bootstrap_status.start(
    "Notebook bootstrap",
    detail="Locating repo root and frozen split artifacts",
)

frozen_splits: dict = {
    partition: load_frozen_split(partition)
    for partition in _PUBLIC_RUNTIME_PARTITIONS
}

_setup_warnings = 0
try:
    PRIVATE_DATASET_ROOT = discover_private_dataset_root()
except FileNotFoundError as _private_err:
    PRIVATE_DATASET_ROOT = None
    _setup_warnings += 1
    _private_dataset_detail = f"private dataset unavailable: {_private_err}"
else:
    if PRIVATE_DATASET_ROOT is not None:
        frozen_splits["private_leaderboard"] = load_private_split(PRIVATE_DATASET_ROOT)
        _private_dataset_detail = f"private dataset attached: {PRIVATE_DATASET_ROOT}"
    else:
        _private_dataset_detail = "private dataset not attached; private_leaderboard skipped"

_partition_summary = ", ".join(
    f"{partition}={len(records)}" for partition, records in frozen_splits.items()
)
_bootstrap_status.finish(
    detail=(
        f"repo root: {REPO_ROOT}\n"
        f"parser={PARSER_VERSION} | metric={METRIC_VERSION}\n"
        f"{_private_dataset_detail}\n"
        f"splits: {_partition_summary}"
    ),
    processed=len(frozen_splits),
    total=len(_PUBLIC_RUNTIME_PARTITIONS) + 1,
    warnings=_setup_warnings,
)
RUN_LOGGER.log_bootstrap_finished(
    detail=(
        f"repo root: {REPO_ROOT}\n"
        f"parser={PARSER_VERSION} | metric={METRIC_VERSION}\n"
        f"{_private_dataset_detail}\n"
        f"splits: {_partition_summary}"
    ),
    processed=len(frozen_splits),
    total=len(_PUBLIC_RUNTIME_PARTITIONS) + 1,
    warnings=_setup_warnings,
)

## Build evaluation dataframes

Builds two separate dataframes:

- `dev_df` — dev split only; local validation, never submitted.
- `leaderboard_df` — `public_leaderboard` and `private_leaderboard` only; fed to the official evaluation path.

In [ ]:
import pandas as pd


_dataframe_status = NotebookStatus(refresh_every=_PROGRESS_REFRESH_EVERY)
_log_phase_start(
    "build_evaluation_frames",
    detail="Preparing dev and leaderboard dataframes",
)
_dataframe_status.start(
    "Building evaluation frames",
    detail="Preparing dev and leaderboard dataframes",
)


def _build_rows(partition: str, records) -> list[dict]:
    rows = []
    for record in records:
        ep = record.episode
        rows.append({
            "episode_id": ep.episode_id,
            "split": partition,
            "difficulty": ep.difficulty.value,
            "template_id": ep.template_id.value,
            "template_family": ep.template_family.value,
            "shift_position": str(ep.shift_after_position),
            "transition_type": ep.transition.value,
            "prompt_binary": render_binary_prompt(ep),
            "prompt_narrative": render_narrative_prompt(ep),
            "probe_targets": tuple(t.value for t in ep.probe_targets),
        })
    return rows


_dev_rows: list[dict] = _build_rows(_DEV_PARTITION, frozen_splits[_DEV_PARTITION])
dev_df = pd.DataFrame(_dev_rows)
assert set(dev_df["split"]) == {_DEV_PARTITION}

_available_leaderboard_partitions = tuple(
    p for p in _LEADERBOARD_PARTITIONS if p in frozen_splits
)
leaderboard_rows = [
    row
    for _partition in _available_leaderboard_partitions
    for row in _build_rows(_partition, frozen_splits[_partition])
]
leaderboard_df = pd.DataFrame(leaderboard_rows)
assert set(leaderboard_df["split"]) == set(_available_leaderboard_partitions)

_BINARY_EVAL_COLUMNS = (
    "episode_id",
    "difficulty",
    "template_id",
    "template_family",
    "shift_position",
    "transition_type",
    "prompt_binary",
    "probe_targets",
)
_NARRATIVE_EVAL_COLUMNS = ("episode_id", "prompt_narrative", "probe_targets")

_episode_split_lookup = {
    row["episode_id"]: row["split"]
    for row in (*_dev_rows, *leaderboard_rows)
}
_leaderboard_episode_lookup = {
    record.episode.episode_id: record.episode
    for _partition in _available_leaderboard_partitions
    for record in frozen_splits[_partition]
}
_binary_parsed_predictions: dict = {}
_narrative_parsed_results: dict = {}

_dataframe_status.finish(
    detail=(
        f"dev_df rows: {len(dev_df)}\n"
        f"leaderboard_df rows: {len(leaderboard_df)}\n"
        f"leaderboard partitions: {', '.join(_available_leaderboard_partitions)}"
    ),
    processed=len(dev_df) + len(leaderboard_df),
    total=len(dev_df) + len(leaderboard_df),
)
_log_phase_finish(
    "build_evaluation_frames",
    detail=(
        f"dev_df rows: {len(dev_df)}\n"
        f"leaderboard_df rows: {len(leaderboard_df)}\n"
        f"leaderboard partitions: {', '.join(_available_leaderboard_partitions)}"
    ),
    processed=len(dev_df) + len(leaderboard_df),
    total=len(dev_df) + len(leaderboard_df),
)
leaderboard_df[["episode_id", "split", "difficulty", "template_id", "template_family", "shift_position", "transition_type"]].head()

## Binary task — official leaderboard task

Registered with `@kbench.task`. `%choose ruleshift_benchmark_v1_binary` selects this as the leaderboard entry. At evaluation time, kbench calls this function once per row in `evaluation_data`, injecting `llm` and matching remaining columns by name. Returns `(num_correct, 4)` per episode.

In [ ]:
@kbench.task(
    name="ruleshift_benchmark_v1_binary",
    description=(
        "Cognitive flexibility benchmark: infer a hidden rule shift from "
        "sparse contradictory evidence in a sequence of charge interactions, "
        "then predict four post-shift probe outcomes."
    ),
)
def ruleshift_benchmark_v1_binary(
    llm,
    prompt_binary: str,
    probe_targets: tuple,
    episode_id: str | None = None,
    template_id: str | None = None,
    template_family: str | None = None,
    difficulty: str | None = None,
    shift_position: str | None = None,
    transition_type: str | None = None,
) -> tuple[int, int]:
    result = run_binary_episode(
        llm=llm,
        prompt_binary=prompt_binary,
        probe_targets=probe_targets,
        logger=RUN_LOGGER,
        ledger=RUN_EPISODE_LEDGER,
        phase=_BINARY_LOG_PHASE,
        task_mode="binary",
        episode_id=episode_id,
        split=_episode_split_lookup.get(episode_id),
    )

    if episode_id is not None:
        _binary_parsed_predictions[episode_id] = result.parsed_prediction

    return result.score

## Narrative function — supplementary companion

Not a `@kbench.task`; not selectable via `%choose`. Results are collected manually and included in the payload as robustness evidence only — they do not affect the leaderboard score.

In [ ]:
def ruleshift_benchmark_v1_narrative(
    llm,
    prompt_narrative: str,
    probe_targets: tuple,
    episode_id: str | None = None,
) -> tuple[int, int]:
    result = run_narrative_episode(
        llm=llm,
        prompt_narrative=prompt_narrative,
        probe_targets=probe_targets,
        logger=RUN_LOGGER,
        ledger=RUN_EPISODE_LEDGER,
        phase=_NARRATIVE_LOG_PHASE,
        task_mode="narrative",
        episode_id=episode_id,
        split=_episode_split_lookup.get(episode_id),
    )

    if episode_id is not None:
        _narrative_parsed_results[episode_id] = result.parsed_result

    return result.score

## Local: scoring sanity checks

Verifies parser and scorer against known inputs using one row from `dev_df`. Local only — does not run on the Kaggle evaluation path.

In [ ]:
sample_row = dev_df.iloc[0]
print(f"Episode: {sample_row['episode_id']}")
print(f"Split: {sample_row['split']}")
print(f"Targets: {sample_row['probe_targets']}")
print()

# Verify scoring with a perfect prediction
perfect = score_episode(sample_row["probe_targets"], sample_row["probe_targets"])
print(f"Perfect prediction score: {perfect}")
assert perfect == (4, 4), f"Expected (4, 4), got {perfect}"

# Verify scoring with an invalid prediction
invalid = score_episode(None, sample_row["probe_targets"])
print(f"Invalid prediction score: {invalid}")
assert invalid == (0, 4), f"Expected (0, 4), got {invalid}"

# Verify normalize with a raw text response
targets_text = ", ".join(sample_row["probe_targets"])
normalized = normalize_binary_response(targets_text)
print(f"Parsed '{targets_text}' -> {normalized}")
assert normalized == sample_row["probe_targets"], f"Parse mismatch"

# Verify structured response path
labels = [Label(t) for t in sample_row["probe_targets"]]
structured = BinaryResponse(*labels)
structured_norm = normalize_binary_response(structured)
print(f"Structured response -> {structured_norm}")
assert structured_norm == sample_row["probe_targets"], f"Structured mismatch"

# Verify malformed text
malformed = normalize_binary_response("I don't know")
malformed_score = score_episode(malformed, sample_row["probe_targets"])
print(f"Malformed score: {malformed_score}")
assert malformed_score == (0, 4), f"Expected (0, 4), got {malformed_score}"

print("\nAll local checks passed.")

## Local: dev split validation

Runs Binary and Narrative against `dev_df` only. Binary receives a signature-aligned subset of columns so local validation matches Kaggle binding behavior. Results are stored in `_dev_*` variables and do not touch `binary_results`, `narrative_results_df`, or the canonical payload. Safe to skip.

In [ ]:
_dev_status = NotebookStatus(refresh_every=_PROGRESS_REFRESH_EVERY)
_dev_total = len(dev_df)
_dev_status.start(
    "Dev validation",
    detail="Running Binary and Narrative on dev only",
    processed=0,
    total=_dev_total * 2,
)

_BINARY_LOG_PHASE = "dev_binary_validation"
_log_phase_start(
    _BINARY_LOG_PHASE,
    task_mode="binary",
    detail="Running Binary validation on dev only",
    total=_dev_total,
)
try:
    _dev_binary_eval_df = dev_df[list(_BINARY_EVAL_COLUMNS)]
    _dev_binary_results = ruleshift_benchmark_v1_binary.evaluate(
        llm=[kbench.llm],
        evaluation_data=_dev_binary_eval_df,
    )
    _dev_binary_df = _dev_binary_results.as_dataframe()
    _dev_status.update(
        detail="Binary validation complete; Narrative validation in progress",
        processed=len(_dev_binary_df),
        total=_dev_total * 2,
        force=True,
    )
    _log_phase_finish(
        _BINARY_LOG_PHASE,
        task_mode="binary",
        detail=f"Binary rows: {len(_dev_binary_df)}",
        processed=len(_dev_binary_df),
        total=_dev_total,
    )
    display(_dev_binary_df.head())
except Exception as _dev_binary_exc:
    _log_exception(_dev_binary_exc, phase=_BINARY_LOG_PHASE, task_mode="binary")
    _dev_status.update(
        detail=f"Binary validation skipped: {_dev_binary_exc}",
        processed=0,
        total=_dev_total * 2,
        warnings=1,
        force=True,
    )
    _log_phase_finish(
        _BINARY_LOG_PHASE,
        task_mode="binary",
        detail=f"Binary validation skipped: {_dev_binary_exc}",
        status="failed",
        level="error",
        processed=0,
        total=_dev_total,
        warnings=1,
    )
    _dev_binary_results = None

_NARRATIVE_LOG_PHASE = "dev_narrative_validation"
_log_phase_start(
    _NARRATIVE_LOG_PHASE,
    task_mode="narrative",
    detail="Running Narrative validation on dev only",
    total=_dev_total,
)
try:
    _dev_nar_rows: list[dict] = []
    for _idx, (_, _row) in enumerate(dev_df.iterrows(), start=1):
        _nc, _tot = ruleshift_benchmark_v1_narrative(
            kbench.llm,
            _row["prompt_narrative"],
            _row["probe_targets"],
            episode_id=_row["episode_id"],
        )
        _dev_nar_rows.append({
            "episode_id": _row["episode_id"],
            "num_correct": _nc,
            "total": _tot,
        })
        _dev_status.update(
            detail="Narrative validation in progress",
            processed=_dev_total + _idx,
            total=_dev_total * 2,
        )
    _dev_narrative_df = pd.DataFrame(_dev_nar_rows)
    _dev_status.finish(
        detail=(
            f"Binary rows: {0 if _dev_binary_results is None else len(_dev_binary_df)}\n"
            f"Narrative rows: {len(_dev_narrative_df)}"
        ),
        processed=_dev_total * 2,
        total=_dev_total * 2,
    )
    _log_phase_finish(
        _NARRATIVE_LOG_PHASE,
        task_mode="narrative",
        detail=f"Narrative rows: {len(_dev_narrative_df)}",
        processed=len(_dev_narrative_df),
        total=_dev_total,
    )
    display(_dev_narrative_df.head())
except Exception as _dev_narrative_exc:
    _log_exception(_dev_narrative_exc, phase=_NARRATIVE_LOG_PHASE, task_mode="narrative")
    _dev_status.finish(
        detail=f"Narrative validation skipped: {_dev_narrative_exc}",
        processed=_dev_total,
        total=_dev_total * 2,
        warnings=1,
    )
    _log_phase_finish(
        _NARRATIVE_LOG_PHASE,
        task_mode="narrative",
        detail=f"Narrative validation skipped: {_dev_narrative_exc}",
        status="failed",
        level="error",
        processed=0,
        total=_dev_total,
        warnings=1,
    )
    _dev_narrative_df = None

## Official Binary evaluation

Calls `.evaluate()` with a Binary-only leaderboard dataframe. The task sees only columns in the Binary task signature, so leaderboard routing and Narrative prompt text never leak into signature binding.

In [ ]:
_binary_parsed_predictions = {}

_BINARY_LOG_PHASE = "official_binary_evaluation"
_binary_status = NotebookStatus(refresh_every=_PROGRESS_REFRESH_EVERY)
_log_phase_start(
    _BINARY_LOG_PHASE,
    task_mode="binary",
    detail="Running official Binary task over leaderboard episodes",
    total=len(leaderboard_df),
)
_binary_status.start(
    "Official Binary evaluation",
    detail="Running official Binary task over leaderboard episodes",
    processed=0,
    total=len(leaderboard_df),
)

try:
    _binary_eval_df = leaderboard_df[list(_BINARY_EVAL_COLUMNS)]
    binary_results = ruleshift_benchmark_v1_binary.evaluate(
        llm=[kbench.llm],
        evaluation_data=_binary_eval_df,
    )
    _binary_results_df = binary_results.as_dataframe()
except Exception as _binary_exc:
    _log_exception(_binary_exc, phase=_BINARY_LOG_PHASE, task_mode="binary")
    _log_phase_finish(
        _BINARY_LOG_PHASE,
        task_mode="binary",
        detail=f"Official Binary evaluation failed: {_binary_exc}",
        status="failed",
        level="error",
        processed=0,
        total=len(leaderboard_df),
        errors=1,
    )
    _invalidate_run(
        phase=_BINARY_LOG_PHASE,
        detail=f"Official Binary evaluation failed: {_binary_exc}",
        reason="official_binary_evaluation_failed",
    )
    raise

_binary_status.finish(
    detail=f"Binary rows produced: {len(_binary_results_df)}",
    processed=len(_binary_results_df),
    total=len(leaderboard_df),
)
_log_phase_finish(
    _BINARY_LOG_PHASE,
    task_mode="binary",
    detail=f"Binary rows produced: {len(_binary_results_df)}",
    processed=len(_binary_results_df),
    total=len(leaderboard_df),
)

## Official Narrative evaluation

Runs the Narrative companion over `leaderboard_df`. Episode-level provider/runtime failures are recorded as operational failures in the durable logs while the phase continues collecting rows.

In [ ]:
_narrative_parsed_results = {}

_NARRATIVE_LOG_PHASE = "official_narrative_evaluation"
_narrative_status = NotebookStatus(refresh_every=_PROGRESS_REFRESH_EVERY)
_log_phase_start(
    _NARRATIVE_LOG_PHASE,
    task_mode="narrative",
    detail="Running supplementary Narrative evaluation",
    total=len(leaderboard_df),
)
_narrative_status.start(
    "Official Narrative evaluation",
    detail="Running supplementary Narrative evaluation",
    processed=0,
    total=len(leaderboard_df),
)

try:
    _nar_eval_df = leaderboard_df[list(_NARRATIVE_EVAL_COLUMNS)]
    _nar_rows: list[dict] = []
    _nar_total = len(_nar_eval_df)
    for _idx, (_, _row) in enumerate(_nar_eval_df.iterrows(), start=1):
        _nc, _tot = ruleshift_benchmark_v1_narrative(
            kbench.llm,
            _row["prompt_narrative"],
            _row["probe_targets"],
            episode_id=_row["episode_id"],
        )
        _nar_rows.append({
            "episode_id": _row["episode_id"],
            "num_correct": _nc,
            "total": _tot,
        })
        _narrative_status.update(
            detail="Collecting Narrative episode scores",
            processed=_idx,
            total=_nar_total,
        )
    narrative_results_df = pd.DataFrame(_nar_rows)
    _narrative_status.finish(
        detail=f"Narrative rows produced: {len(narrative_results_df)}",
        processed=len(narrative_results_df),
        total=_nar_total,
    )
    _log_phase_finish(
        _NARRATIVE_LOG_PHASE,
        task_mode="narrative",
        detail=f"Narrative rows produced: {len(narrative_results_df)}",
        processed=len(narrative_results_df),
        total=_nar_total,
    )
except Exception as _narrative_exc:
    _log_exception(_narrative_exc, phase=_NARRATIVE_LOG_PHASE, task_mode="narrative")
    _narrative_status.finish(
        detail=f"Narrative evaluation skipped: {_narrative_exc}",
        processed=0,
        total=len(leaderboard_df),
        warnings=1,
    )
    _log_phase_finish(
        _NARRATIVE_LOG_PHASE,
        task_mode="narrative",
        detail=f"Narrative evaluation skipped: {_narrative_exc}",
        status="failed",
        level="error",
        processed=0,
        total=len(leaderboard_df),
        warnings=1,
    )
    narrative_results_df = None

## Diagnostics

Summarizes parser validity, Binary and Narrative accuracy, and canonical slice diagnostics over the same leaderboard episodes used for the official payload.

In [ ]:
_diagnostics_status = NotebookStatus(refresh_every=_PROGRESS_REFRESH_EVERY)
_log_phase_start(
    "diagnostics",
    detail="Building parser, metric, and slice diagnostics from collected leaderboard results",
    total=3,
)
_diagnostics_status.start(
    "Diagnostics",
    detail="Building parser, metric, and slice diagnostics from collected leaderboard results",
    processed=0,
    total=3,
)

if narrative_results_df is None:
    _log_phase_finish(
        "diagnostics",
        detail="Narrative diagnostics require narrative_results_df to be present",
        status="failed",
        level="error",
        processed=0,
        total=3,
        errors=1,
    )
    _invalidate_run(
        phase="diagnostics",
        detail="Narrative diagnostics require narrative_results_df to be present",
        reason="narrative_results_df_missing",
    )
    raise ValueError("Narrative diagnostics require narrative_results_df to be present")

_binary_prediction_sequence = tuple(
    _binary_parsed_predictions[episode_id]
    for episode_id in leaderboard_df["episode_id"]
)
_narrative_result_sequence = tuple(
    _narrative_parsed_results[episode_id]
    for episode_id in leaderboard_df["episode_id"]
)
_binary_target_sequence = tuple(
    _leaderboard_episode_lookup[episode_id].probe_targets
    for episode_id in leaderboard_df["episode_id"]
)
metric_summary = compute_metrics(
    _binary_prediction_sequence,
    _binary_target_sequence,
    _narrative_result_sequence,
)
_diagnostics_status.update(
    detail="Computed Binary and Narrative metric summaries",
    processed=1,
    total=3,
    force=True,
)

slice_report = build_slice_report(
    tuple(
        compute_episode_slice_data(
            episode=_leaderboard_episode_lookup[episode_id],
            prediction=_binary_parsed_predictions[episode_id],
            narrative_result=_narrative_parsed_results.get(episode_id),
        )
        for episode_id in leaderboard_df["episode_id"]
    )
)
_slice_report_dict = slice_report.to_dict()
_diagnostics_status.update(
    detail="Computed canonical leaderboard slice report",
    processed=2,
    total=3,
    force=True,
)

_narrative_denominator = int(narrative_results_df["total"].sum())
diagnostics_summary = {
    "binary_accuracy": metric_summary.post_shift_probe_accuracy,
    "narrative_accuracy": (
        int(narrative_results_df["num_correct"].sum()) / _narrative_denominator
        if _narrative_denominator > 0
        else 0.0
    ),
    "binary_parse_valid_rate": metric_summary.binary_parse_valid_rate,
    "narrative_schema_valid_rate": metric_summary.narrative_schema_valid_rate,
    "narrative_parse_failure_count": metric_summary.narrative_parse_failure_count,
}
display(pd.DataFrame([diagnostics_summary]))

diagnostic_slice_frames: dict[str, pd.DataFrame] = {}
for _slice_name in ("difficulty", "template_family", "shift_position", "transition_type"):
    _frame = pd.DataFrame.from_dict(_slice_report_dict[_slice_name], orient="index")
    _frame.index.name = _slice_name
    diagnostic_slice_frames[_slice_name] = _frame.reset_index()
    display(diagnostic_slice_frames[_slice_name])

diagnostic_error_type_df = pd.DataFrame(
    [{"error_type": key, "count": value} for key, value in _slice_report_dict["error_type"].items()]
)
display(diagnostic_error_type_df)
_diagnostics_status.finish(
    detail=(
        f"Binary accuracy: {diagnostics_summary['binary_accuracy']:.4f}\n"
        f"Narrative accuracy: {diagnostics_summary['narrative_accuracy']:.4f}\n"
        f"Binary parse valid rate: {diagnostics_summary['binary_parse_valid_rate']:.4f}\n"
        f"Narrative schema valid rate: {diagnostics_summary['narrative_schema_valid_rate']:.4f}"
    ),
    processed=3,
    total=3,
)
_log_phase_finish(
    "diagnostics",
    detail=(
        f"Binary accuracy: {diagnostics_summary['binary_accuracy']:.4f}\n"
        f"Narrative accuracy: {diagnostics_summary['narrative_accuracy']:.4f}\n"
        f"Binary parse valid rate: {diagnostics_summary['binary_parse_valid_rate']:.4f}\n"
        f"Narrative schema valid rate: {diagnostics_summary['narrative_schema_valid_rate']:.4f}"
    ),
    processed=3,
    total=3,
)

## Canonical payload

Assembles and validates the official result payload from `binary_results` and `narrative_results_df`. The `print()` output is the authoritative Kaggle-emitted artifact. Both inputs are required — missing either raises an error.

In [ ]:
import json
from core.kaggle import (
    build_kaggle_payload,
    normalize_count_result_df,
    validate_kaggle_payload,
)

_payload_status = NotebookStatus(refresh_every=_PROGRESS_REFRESH_EVERY)
_log_phase_start(
    "canonical_payload",
    detail="Normalizing evaluation outputs and validating final artifact",
    total=3,
)
_payload_status.start(
    "Canonical payload",
    detail="Normalizing evaluation outputs and validating final artifact",
    processed=0,
    total=3,
)

if 'binary_results' not in locals() or binary_results is None:
    _log_phase_finish(
        "canonical_payload",
        detail="CRITICAL: binary_results is missing. Binary evaluation did not run or failed before producing results.",
        status="failed",
        level="error",
        processed=0,
        total=3,
        errors=1,
    )
    _invalidate_run(
        phase="canonical_payload",
        detail="CRITICAL: binary_results is missing. Binary evaluation did not run or failed before producing results.",
        reason="binary_results_missing",
    )
    raise ValueError(
        "CRITICAL: binary_results is missing. "
        "Binary evaluation did not run or failed before producing results."
    )

if 'narrative_results_df' not in locals() or narrative_results_df is None:
    _log_phase_finish(
        "canonical_payload",
        detail="CRITICAL: narrative_results_df is missing. Narrative evaluation is mandatory for a valid release.",
        status="failed",
        level="error",
        processed=0,
        total=3,
        errors=1,
    )
    _invalidate_run(
        phase="canonical_payload",
        detail="CRITICAL: narrative_results_df is missing. Narrative evaluation is mandatory for a valid release.",
        reason="narrative_results_df_missing",
    )
    raise ValueError(
        "CRITICAL: narrative_results_df is missing. "
        "Narrative evaluation is mandatory for a valid release. "
        "Both Binary and Narrative must run on the same frozen episodes."
    )

binary_df_payload = normalize_count_result_df(
    binary_results.as_dataframe(),
    dataframe_label="binary_df",
)
narrative_df_payload = normalize_count_result_df(
    narrative_results_df,
    dataframe_label="narrative_df",
)
_payload_status.update(
    detail="Normalized Binary and Narrative outputs",
    processed=1,
    total=3,
    force=True,
)

payload = build_kaggle_payload(
    binary_df=binary_df_payload,
    narrative_df=narrative_df_payload,
    slice_report=slice_report,
)
_payload_status.update(
    detail="Payload assembled; running contract validation",
    processed=2,
    total=3,
    force=True,
)
validate_kaggle_payload(payload)
RUN_LOGGER.log_payload_built(
    total_episodes=payload["primary_result"]["total_episodes"],
    binary_score=payload["primary_result"]["score"],
    narrative_score=payload["narrative_result"]["score"],
)
_payload_status.finish(
    detail=(
        f"Validated payload with {payload['primary_result']['total_episodes']} episodes\n"
        f"binary score: {payload['primary_result']['score']:.4f} | narrative score: {payload['narrative_result']['score']:.4f}"
    ),
    processed=3,
    total=3,
)
_log_phase_finish(
    "canonical_payload",
    detail=(
        f"Validated payload with {payload['primary_result']['total_episodes']} episodes\n"
        f"binary score: {payload['primary_result']['score']:.4f} | narrative score: {payload['narrative_result']['score']:.4f}"
    ),
    processed=3,
    total=3,
)

# Public result: the print output below is the authoritative Kaggle-emitted artifact.
print("=== RuleShift Benchmark v1 — Canonical Result ===")
print(json.dumps(payload, indent=2))
print("=== End Canonical Result ===")

# Optional debug side file — not the authoritative result.
try:
    with BENCHMARK_RESULT_PATH.open("w", encoding="utf-8") as _f:
        json.dump(payload, _f, indent=2)
except Exception as _benchmark_result_exc:
    RUN_LOGGER.log(
        phase="canonical_payload",
        event="artifact_write",
        level="warning",
        status="failed",
        task_mode="notebook",
        episode_id=None,
        artifact_path=str(BENCHMARK_RESULT_PATH),
        detail=f"benchmark_result.json write skipped: {_benchmark_result_exc}",
    )
else:
    RUN_LOGGER.log(
        phase="canonical_payload",
        event="artifact_write",
        level="info",
        status="completed",
        task_mode="notebook",
        episode_id=None,
        artifact_path=str(BENCHMARK_RESULT_PATH),
    )

_exception_summary = RUN_LOGGER.summarize_exceptions()

RUN_LOGGER.log_run_finished(
    output_dir=str(RUN_OUTPUT_DIR),
    total_episodes=payload["primary_result"]["total_episodes"],
    total_exceptions=_exception_summary.total,
)

try:
    DIAGNOSTICS_SUMMARY_PATH = write_diagnostics_summary(
        context=RUN_CONTEXT,
        binary_parse_valid_rate=(
            metric_summary.binary_parse_valid_rate
            if 'metric_summary' in locals()
            else None
        ),
        narrative_schema_valid_rate=(
            metric_summary.narrative_schema_valid_rate
            if 'metric_summary' in locals()
            else None
        ),
    )
except Exception as _diagnostics_summary_exc:
    RUN_LOGGER.log(
        phase="run",
        event="artifact_write",
        level="warning",
        status="failed",
        task_mode="notebook",
        episode_id=None,
        artifact_path=str(RUN_OUTPUT_DIR / "diagnostics_summary.json"),
        detail=f"diagnostics_summary.json write skipped: {_diagnostics_summary_exc}",
    )
else:
    RUN_LOGGER.log(
        phase="run",
        event="artifact_write",
        level="info",
        status="completed",
        task_mode="notebook",
        episode_id=None,
        artifact_path=str(DIAGNOSTICS_SUMMARY_PATH),
    )

try:
    RUN_MANIFEST_PATH = write_run_manifest(
        context=RUN_CONTEXT,
        repo_root=REPO_ROOT,
    )
except Exception as _run_manifest_exc:
    RUN_LOGGER.log(
        phase="run",
        event="artifact_write",
        level="warning",
        status="failed",
        task_mode="notebook",
        episode_id=None,
        artifact_path=str(RUN_OUTPUT_DIR / "run_manifest.json"),
        detail=f"run_manifest.json write skipped: {_run_manifest_exc}",
    )
else:
    RUN_LOGGER.log(
        phase="run",
        event="artifact_write",
        level="info",
        status="completed",
        task_mode="notebook",
        episode_id=None,
        artifact_path=str(RUN_MANIFEST_PATH),
    )


## Task selection

Selects `ruleshift_benchmark_v1_binary` as the single leaderboard task. Kaggle Benchmarks uses `%choose` to determine which task result is submitted.

In [ ]:
%choose ruleshift_benchmark_v1_binary